In [1]:
import psycopg2
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import roc_auc_score

In [ ]:
# Connect to database
conn = psycopg2.connect(
    host="ep-billowing-sunset-ab0xh9uf-pooler.eu-west-2.aws.neon.tech",
    database="neondb",
    user="neondb_owner",
    password="*******",
    port="5432",
    sslmode="require"
)

cursor = conn.cursor()
query = "SELECT * FROM transactions"
FD = pd.read_sql(query, conn)

FD.head()

C:\Users\ty\AppData\Local\Temp\ipykernel_7732\217683509.py:13: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  FD = pd.read_sql(query, conn)


,transaction_id,customer_id,transaction_amount,transaction_time,location,ip_address,device_id,is_foreign_transaction,device_change_flag,is_fraud
0,926790,20,2983.03,2026-04-09 05:02:05.665239,Abuja,254.238.155.162,tablet,1,0,0
1,496382,1,38407.71,2026-04-09 05:02:05.665239,Lagos,130.40.133.139,tablet,0,0,0
2,943913,39,49098.51,2026-04-09 05:02:05.665239,Port Harcourt,57.238.129.9,web,0,0,1
3,180877,47,22249.60,2026-04-09 05:02:05.665239,Lagos,38.59.150.36,mobile,1,0,1
4,948234,38,3144.81,2026-04-09 05:02:05.665239,Lagos,43.70.38.212,mobile,0,0,0


In [3]:
# REPROCESSING

# Convert time
FD['transaction_time'] = pd.to_datetime(FD['transaction_time'])

# Sort for time-based features
FD = FD.sort_values(['customer_id', 'transaction_time'])

In [4]:
# Feature engineering
FD['time_diff'] = FD.groupby('customer_id')['transaction_time'].diff().dt.total_seconds()
FD['time_diff'] = FD['time_diff'].fillna(9999)

FD['transaction_hour'] = FD['transaction_time'].dt.hour

FD['txn_count'] = FD.groupby('customer_id').cumcount() + 1

FD['avg_amount'] = FD.groupby('customer_id')['transaction_amount'].transform('mean')

FD['relative_amount'] = FD['transaction_amount'] / (FD['avg_amount'] + 1)


In [5]:
# Drop unused/high cardinality
FD = FD.drop(columns=['ip_address'])

# Encode categorical features
FD = pd.get_dummies(FD, columns=['location', 'device_id'], drop_first=True)

In [6]:
# Features & target
X = FD.drop(columns=['is_fraud', 'transaction_time', 'transaction_id'])
y = FD['is_fraud']

In [7]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y)

In [8]:
# Model
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=12,
    min_samples_leaf=3,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

In [9]:
# Train
model.fit(X_train, y_train)

,n_estimators,300
,criterion,'gini'
,max_depth,12
,min_samples_split,2
,min_samples_leaf,3
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [10]:
# Evaluation
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

roc = roc_auc_score(y_test, y_proba)
print("\nROC-AUC:", roc)

Confusion Matrix:
[[659 116]
 [ 73 152]]

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.85      0.87       775
           1       0.57      0.68      0.62       225

    accuracy                           0.81      1000
   macro avg       0.73      0.76      0.75      1000
weighted avg       0.83      0.81      0.82      1000


ROC-AUC: 0.8562637992831541


In [12]:
import numpy as np
y_proba = model.predict_proba(X_test)[:, 1]

threshold = [0.3, 0.4, 0.5, 0.6, 0.7]

for t in threshold:
    print(f"\nThreshold: {t}")
    
    y_pred = (y_proba >= t).astype(int)
    
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))


Threshold: 0.3
[[566 209]
 [ 32 193]]
              precision    recall  f1-score   support

           0       0.95      0.73      0.82       775
           1       0.48      0.86      0.62       225

    accuracy                           0.76      1000
   macro avg       0.71      0.79      0.72      1000
weighted avg       0.84      0.76      0.78      1000


Threshold: 0.4
[[597 178]
 [ 41 184]]
              precision    recall  f1-score   support

           0       0.94      0.77      0.85       775
           1       0.51      0.82      0.63       225

    accuracy                           0.78      1000
   macro avg       0.72      0.79      0.74      1000
weighted avg       0.84      0.78      0.80      1000


Threshold: 0.5
[[659 116]
 [ 73 152]]
              precision    recall  f1-score   support

           0       0.90      0.85      0.87       775
           1       0.57      0.68      0.62       225

    accuracy                           0.81      1000
   macro av

In [ ]:
import joblib
import json

joblib.dump(model, "C:/Users/ty/OneDrive/Pictures/Documents/Fraud Detection Project/Near Live Simulation of Fraud Detection/Model/fraud_model.pkl")

with open("C:/Users/ty/OneDrive/Pictures/Documents/Fraud Detection Project/Near Live Simulation of Fraud Detection/Model/threshold.json", "w") as f:
    json.dump({"threshold": 0.4}, f)